In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

In [ ]:
!cp -r /content/drive/MyDrive/humpback-whale-identification /content/humpback

In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c 'humpback-whale-identification'
!unzip -q humpback-whale-identification.zip -d /content/humpback

In [ ]:
import os

# 1. Clean up any previous failed attempts
!rm -rf /content/humpback

# 2. Create the destination directory
!mkdir -p /content/humpback

# 3. Copy the ZIP file from Drive to Colab VM (Fast)
# Change 'humpback-whale-identification.zip' to your actual zip filename if different
!cp "/content/drive/MyDrive/humpback-whale-identification.zip" /content/humpback_data.zip

# 4. Unzip directly into the target folder (Fast)
!unzip -q /content/humpback_data.zip -d /content/humpback

# 5. Clean up the zip file to save space
!rm /content/humpback_data.zip

print("Data successfully unzipped to /content/humpback")
print(f"Files in folder: {len(os.listdir('/content/humpback'))}")

In [ ]:
# imports
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
import random
import pandas as pd
import seaborn as sns
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
import torch.nn.functional as F
from torchvision import models, transforms
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from tqdm import tqdm
from torch.utils.data import DataLoader

# constants
TRAIN_DIR = "/content/humpback/train"
CSV_PATH = "/content/humpback/train.csv"
NUM_CLASSES_TO_USE = 15
BATCH_SIZE = 32
NUM_EPOCHS = 20
K_FOLDS = 5

In [ ]:
# handling the data
full_df = pd.read_csv(CSV_PATH)
top_classes = full_df['Id'].value_counts().index[1:NUM_CLASSES_TO_USE+1]
df = full_df[full_df['Id'].isin(top_classes)].reset_index(drop=True)

# Extract one category for question 2e
chosen_id = "w_23a388d"
df_chosen = df[df["Id"] == chosen_id]
df_other = df[df["Id"] != chosen_id].copy()

# Map class IDs to numerical labels
class_to_idx = {cls_name: i for i, cls_name in enumerate(top_classes)}
df_other = df_other.copy()
df_other['label_idx'] = df_other['Id'].map(class_to_idx)
df['label_idx'] = df['Id'].map(class_to_idx)

# Custom Dataset Class
class WhaleDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe.copy()
        self.root_dir = root_dir
        self.transform = transform


    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['Image']
        label = row['label_idx']
        img_path = os.path.join(self.root_dir, img_name)

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label


# Part 1

**1a**

In [ ]:
# Basic Data Stats
print(f"Total number of training images: {len(full_df)}")
print(f"Total unique whale classes (IDs): {full_df['Id'].nunique()}")

# Check for Class Imbalance
class_counts = full_df['Id'].value_counts()
print(f"\nMost common whale: {class_counts.index[0]} with {class_counts.iloc[0]} images")
print(f"Least common whale: {class_counts.index[-1]} with {class_counts.iloc[-1]} images")

# Visualize the Distribution (Top 20 classes)
plt.figure(figsize=(12, 6))
sns.barplot(x=class_counts[:20].index, y=class_counts[:20].values)
plt.xticks(rotation=90)
plt.title("Top 20 Most Common Whale IDs (Check for Imbalance)")
plt.xlabel("Whale ID")
plt.ylabel("Number of Images")
plt.show()

# Check Image Dimensions (Preprocessing needs)
# We check the shape of the first 100 images to see if they vary
shapes = []
for img_name in full_df['Image'].iloc[:100]:
    img_path = os.path.join(TRAIN_DIR, img_name)
    with Image.open(img_path) as img:
        shapes.append(img.size) # (width, height)

print(f"\nSample of image sizes (Width, Height): {shapes[:5]}")

**1b.**
Each sample contains an image and a label. The images have different sizes so we should resize them to be in the same size. The images were taken from different angles and in different lighting so we can augment the data, the model should work well on the augmented data.

**1c.** As we can see above, the dataset is extremely imbalanced, as there's almost 10000 whales as "new whale" which means they are unidentified, while specific whales have only a couple of images because of that we will use the 15 classes with the most samples excluding new whale.

In [ ]:
# Basic Data Stats
print(f"Total number of training images: {len(df_other)}")
print(f"Total unique whale classes (IDs): {df_other['Id'].nunique()}")

# Check for Class Imbalance
class_counts = df_other['Id'].value_counts()
print(f"\nMost common whale: {class_counts.index[0]} with {class_counts.iloc[0]} images")
print(f"Least common whale: {class_counts.index[-1]} with {class_counts.iloc[-1]} images")

# Visualize the Distribution
plt.figure(figsize=(12, 6))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.xticks(rotation=90)
plt.title("Most Common Whale IDs")
plt.xlabel("Whale ID")
plt.ylabel("Number of Images")
plt.show()

# Check Image Dimensions (Preprocessing needs)
# We check the shape of the first 100 images to see if they vary
shapes = []
for img_name in df_other['Image'].iloc[:100]:
    img_path = os.path.join(TRAIN_DIR, img_name)
    with Image.open(img_path) as img:
        shapes.append(img.size) # (width, height)

print(f"\nSample of image sizes (Width, Height): {shapes[:5]}")

**1d**
In the Kaggle leaderboard there are contestants that have achieved a score of about 97% accuracy they used transfer learning with the senet154 model and they did 4 fold cross validation.

**1e**

In [ ]:
NUM_SAMPLES_PER_CLASS = 3

# Group images by label
label_to_images = defaultdict(list)
for idx, row in df_other.iterrows():
    label_to_images[row['Id']].append(row['Image'])

# Pick a few labels to visualize (if too many)
selected_labels = list(label_to_images.keys())[:10]  # adjust 10 to more/fewer

plt.figure(figsize=(15, len(selected_labels) * 3))

for i, label in enumerate(selected_labels):
    images = label_to_images[label][:NUM_SAMPLES_PER_CLASS]  # pick a few
    for j, img_name in enumerate(images):
        img_path = os.path.join(TRAIN_DIR, img_name)
        img = Image.open(img_path).convert("RGB")

        ax = plt.subplot(len(selected_labels), NUM_SAMPLES_PER_CLASS, i*NUM_SAMPLES_PER_CLASS + j + 1)
        plt.imshow(img)
        if j == 0:
            ax.set_ylabel(label, fontsize=12)
        plt.xticks([])
        plt.yticks([])

plt.suptitle("Samples from each whale label (top 10 shown)", fontsize=16)
plt.tight_layout()
plt.show()

# Part 2

In [ ]:
print(f"Training on subset: {len(df_other)} images, {len(top_classes)} classes.")

# resizing the images
transform = transforms.Compose([
    transforms.Resize((128, 128)), # Smaller size for speed
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

dataset = WhaleDataset(df_other, TRAIN_DIR, transform=transform)

# define custom model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)

        # Conv Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Conv Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Fully Connected Layers
        self.flatten = nn.Flatten()
        # 128 channels * 16 * 16 (image size after 3 pools of 128x128)
        self.fc1 = nn.Linear(128 * 16 * 16, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

#  K-fold cross validation loop
kfold = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Store results for final report
fold_results = {}

labels = df_other['label_idx'].tolist()

for fold, (train_ids, val_ids) in enumerate(kfold.split(df_other, labels)):
    print(f"\n--- FOLD {fold+1}/{K_FOLDS} ---")

    # Init DataLoaders for this fold
    train_sub = Subset(dataset, train_ids)
    val_sub = Subset(dataset, val_ids)

    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False)

    # Initial Model
    model = SimpleCNN(num_classes=len(top_classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    history = {'train_loss': [], 'val_acc': []}

    # Training Loop
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        # Validation Loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        history['val_acc'].append(val_acc)

        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.2f}%")

    fold_results[fold] = history

print("FINAL CROSS-VALIDATION RESULTS")


final_fold_acc = []
final_fold_loss = []

for fold in fold_results:
    fold_acc = fold_results[fold]['val_acc'][-1]       # last epoch accuracy
    fold_loss = fold_results[fold]['train_loss'][-1]   # last epoch loss
    final_fold_acc.append(fold_acc)
    final_fold_loss.append(fold_loss)
    print(f"Fold {fold+1}: Accuracy = {fold_acc:.2f}%, Loss = {fold_loss:.4f}")

# Averages
mean_acc = np.mean(final_fold_acc)
mean_loss = np.mean(final_fold_loss)

print(f"Mean Cross-Validation Accuracy: {mean_acc:.4f}%")
print(f"Mean Cross-Validation Loss: {mean_loss:.4f}")

# --- 4. VISUALIZATION  ---
print("\nPlotting Results...")
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
for fold in fold_results:
    plt.plot(fold_results[fold]['train_loss'], label=f'Fold {fold+1}')
plt.title("Training Loss per Fold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
for fold in fold_results:
    plt.plot(fold_results[fold]['val_acc'], label=f'Fold {fold+1}')
plt.title("Validation Accuracy per Fold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.show()

###2b

In [ ]:
# Make a list of filenames for validation set
val_filenames = [df_other.iloc[i]['Image'] for i in val_ids]

misclassified = []
predictions = []

pointer = 0  # to track position in val_filenames

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        predictions.extend(predicted.cpu().numpy())

        batch_size = labels.size(0)
        for i in range(batch_size):
            true_label = labels[i].item()
            pred_label = predicted[i].item()
            img_name = val_filenames[pointer + i]

            if pred_label != true_label:
                img_path = os.path.join(TRAIN_DIR, img_name)
                raw_img = Image.open(img_path).convert("RGB")
                misclassified.append((raw_img, true_label, pred_label, img_name))

        pointer += batch_size

print(f"Total misclassified: {len(misclassified)}")


In [ ]:
#Displaying some misclassified images
import matplotlib.pyplot as plt

num_show = min(10, len(misclassified))  # show up to 10

plt.figure(figsize=(15, 5))
for i in range(num_show):
    img, true_label, pred_label, filename = misclassified[i]
    plt.subplot(2, 5, i+1)
    plt.imshow(img)
    plt.title(f"True: {true_label} | Pred: {pred_label}")
    plt.axis("off")
plt.show()

In [ ]:
#Displaying some of the correct images
# Make a list of filenames for validation set
val_filenames = [df_other.iloc[i]['Image'] for i in val_ids]

correct = []
predictions = []

pointer = 0  # to track position in val_filenames

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        predictions.extend(predicted.cpu().numpy())

        batch_size = labels.size(0)
        for i in range(batch_size):
            true_label = labels[i].item()
            pred_label = predicted[i].item()
            img_name = val_filenames[pointer + i]

            if pred_label == true_label:
                img_path = os.path.join(TRAIN_DIR, img_name)
                raw_img = Image.open(img_path).convert("RGB")
                correct.append((raw_img, true_label, pred_label, img_name))

        pointer += batch_size

num_show = min(10, len(correct))  # show up to 10

plt.figure(figsize=(15, 5))
for i in range(num_show):
    img, true_label, pred_label, filename = correct[i]
    plt.subplot(2, 5, i+1)
    plt.imshow(img)
    plt.title(f"True: {true_label} | Pred: {pred_label}")
    plt.axis("off")
plt.show()

There are several reasons why the model misclassifies images
1. In some images the lighting is less good
2. In some images the tail is very small compared to the ocean
3. Some of the different tail types are very similair
4. The tails of the whale are sometime showen from different angles

**2c**

3 ways to improve our model (Prioritized)
1. We can add RandomRotation library to help with tilted tails
2. We can use ColorJitter library to help with the different lighting of the images
3. We can crop the images if the tail is only in a small part of the image

In [ ]:
# Data augmentation
transform = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

dataset = WhaleDataset(df_other, TRAIN_DIR, transform=transform)

# CUSTOM CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2,2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128*16*16, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# training loop
kfold = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fold_results = {}
labels_list = df_other['label_idx'].tolist()

print(f"Starting training on device: {device}")

for fold, (train_ids, val_ids) in enumerate(kfold.split(df_other, labels_list)):
    print(f"\n--- AUGMENTED RUN | FOLD {fold+1}/{K_FOLDS} ---")

    train_sub = Subset(dataset, train_ids)
    val_sub = Subset(dataset, val_ids)
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False)

    num_classes = df_other['label_idx'].max() + 1  # FIX: ensure output layer matches labels
    model = SimpleCNN(num_classes=num_classes).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    history = {'train_loss': [], 'val_acc': []}

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data,1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        history['val_acc'].append(val_acc)
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Val Acc {val_acc:.2f}%")

    fold_results[fold] = history

final_fold_acc = []
final_fold_loss = []

for fold in fold_results:
    fold_acc = fold_results[fold]['val_acc'][-1]       # last epoch accuracy
    fold_loss = fold_results[fold]['train_loss'][-1]   # last epoch loss
    final_fold_acc.append(fold_acc)
    final_fold_loss.append(fold_loss)

# Averages
mean_acc = np.mean(final_fold_acc)
mean_loss = np.mean(final_fold_loss)
print("FINAL CROSS-VALIDATION RESULTS")
print(f"Mean Cross-Validation Accuracy: {mean_acc:.4f}%")
print(f"Mean Cross-Validation Loss: {mean_loss:.4f}")

# Plot results
plt.figure(figsize=(8,4))
for fold in fold_results:
    plt.plot(fold_results[fold]['val_acc'], label=f'Fold {fold+1}')
plt.title("Part 2C: Custom CNN with Data Augmentation")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



Implementing data augmentation decreased validation accuracy after 20 epochs. This suggests that the augmentation acted as a strong regularizer, making the training task more difficult. Since the task is harder, it takes the model slower to train. However, it is possible that by running more epochs we would achieve better results.

**2d**

In [ ]:
N_TTA_AUGMENTATIONS = 5
model.eval()
criterion = nn.CrossEntropyLoss()

print(f"--- PART 2d: INFERENCE TIME AUGMENTATION (TTA) ---")
print(f"Aggregating {N_TTA_AUGMENTATIONS} predictions per image...")

correct_standard = 0
correct_tta = 0
total = 0

loss_standard_total = 0.0
loss_tta_total = 0.0

print(f"Processing {len(val_ids)} validation images...")

with torch.no_grad():
    for idx in val_ids:

        # Standart prediction
        img, label = dataset[idx]
        img_batch = img.unsqueeze(0).to(device)
        label_tensor = torch.tensor([label]).to(device)

        output = model(img_batch)

        # Standard loss
        loss_standard = criterion(output, label_tensor)
        loss_standard_total += loss_standard.item()

        _, pred_standard = torch.max(output, 1)
        if pred_standard.item() == label:
            correct_standard += 1

        # TTA prediction
        aug_images = []
        for _ in range(N_TTA_AUGMENTATIONS):
            img_aug, _ = dataset[idx]
            aug_images.append(img_aug)

        aug_batch = torch.stack(aug_images).to(device)
        outputs_aug = model(aug_batch)

        # Compute TTA loss by averaging individual losses
        label_repeat = torch.tensor([label] * N_TTA_AUGMENTATIONS).to(device)
        losses_aug = criterion(outputs_aug, label_repeat)
        loss_tta_total += losses_aug.item()

        # Convert logits → probabilities
        probs_aug = F.softmax(outputs_aug, dim=1)

        # Average the probability vectors
        avg_probs = torch.mean(probs_aug, dim=0)
        _, pred_tta = torch.max(avg_probs, 0)

        if pred_tta.item() == label:
            correct_tta += 1

        total += 1

        if total % 100 == 0:
            print(f"Processed {total} images...")

# calculating results
acc_standard = 100 * correct_standard / total
acc_tta = 100 * correct_tta / total
improvement = acc_tta - acc_standard

avg_loss_standard = loss_standard_total / total
avg_loss_tta = loss_tta_total / total

print("\n" + "="*40)
print("RESULTS FOR PART 2d")
print("="*40)
print(f"Standard Accuracy: {acc_standard:.2f}%")
print(f"TTA Accuracy ({N_TTA_AUGMENTATIONS}x): {acc_tta:.2f}%")
print("-" * 40)
print(f"IMPROVEMENT: {improvement:+.2f}%")
print("="*40)
print(f"Standard Avg Loss: {avg_loss_standard:.4f}")
print(f"TTA Avg Loss:      {avg_loss_tta:.4f}")
print("="*40)

We recieved a small improvement of 1.27% over the single-shot prediction. However, due to the short training time we still have'nt improven over the baseline which had the simplier mission

**2e**

In [ ]:
# This adds Rotation and Color Jitter to fix the errors you found
improved_transform = transforms.Compose([
    transforms.RandomRotation(degrees=10),          # Improvement 1: Fix geometric issues
    transforms.ColorJitter(brightness=0.1, contrast=0.1), # Improvement 2: Fix lighting
    transforms.Resize((128, 128)),                  # Mandatory
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])



# Define custom model (Redefined to be safe)
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2) # 128 -> 64

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2) # 64 -> 32 (Applied twice in forward)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Flatten size calculation:
        # Input 128x128.
        # After Pool 1: 64x64.
        # After Pool 2: 32x32.
        # After Pool 3: 16x16.
        # Final Channels: 128.
        # Flattened = 128 * 16 * 16
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 16 * 16, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        # We define pool explicitly here to match the layers above
        pool = nn.MaxPool2d(2, 2)
        x = pool(self.relu(self.bn1(self.conv1(x))))
        x = pool(self.relu(self.bn2(self.conv2(x))))
        x = pool(self.relu(self.bn3(self.conv3(x))))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Run training loop
dataset = WhaleDataset(df, TRAIN_DIR, transform=improved_transform)
kfold = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fold_results_aug = {}
labels_list = df['label_idx'].tolist()

print(f"Starting Training on {device}...")

for fold, (train_ids, val_ids) in enumerate(kfold.split(df, labels_list)):
    print(f"\n--- AUGMENTED RUN | FOLD {fold+1}/{K_FOLDS} ---")

    train_sub = Subset(dataset, train_ids)
    val_sub = Subset(dataset, val_ids)
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False)

    # Dynamically get class count to match df
    num_classes = df['label_idx'].nunique()
    model = SimpleCNN(num_classes=num_classes).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    history = {'train_loss': [], 'val_acc': []}

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        history['val_acc'].append(val_acc)
        print(f"Ep {epoch+1}: Val Acc {val_acc:.2f}%")

    fold_results_aug[fold] = history

final_fold_acc = []
final_fold_loss = []

for fold in fold_results_aug:
    fold_acc = fold_results_aug[fold]['val_acc'][-1]       # last epoch accuracy
    fold_loss = fold_results_aug[fold]['train_loss'][-1]   # last epoch loss
    final_fold_acc.append(fold_acc)
    final_fold_loss.append(fold_loss)

# Averages
mean_acc = np.mean(final_fold_acc)
mean_loss = np.mean(final_fold_loss)
print("FINAL CROSS-VALIDATION RESULTS")
print(f"Mean Cross-Validation Accuracy: {mean_acc:.4f}%")
print(f"Mean Cross-Validation Loss: {mean_loss:.4f}")

# Plot results
plt.figure(figsize=(8, 4))
for fold in fold_results_aug:
    plt.plot(fold_results_aug[fold]['val_acc'], label=f'Fold {fold+1}')
plt.title("Part 2e: Adding a category")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Part 3

**3a**

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split


# SETTINGS
EPOCHS_PHASE_1 = 5
EPOCHS_PHASE_2 = 5
LR_PHASE_1 = 0.001
LR_PHASE_2 = 0.0001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# train / test split (10% test)
train_df, test_df = train_test_split(
    df,
    test_size=0.10,
    stratify=df["label_idx"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_dataset = WhaleDataset(train_df, TRAIN_DIR, transform=transform)
test_dataset = WhaleDataset(test_df, TRAIN_DIR, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# MODEL BUILDER
def get_model_and_optimizer(model_name, num_classes, device):
    weights = "DEFAULT"

    if model_name == "resnet18":
        model = models.resnet18(weights=weights)
        for p in model.parameters():
            p.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        params = model.fc.parameters()

    elif model_name == "densenet121":
        model = models.densenet121(weights=weights)
        for p in model.parameters():
            p.requires_grad = False
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
        params = model.classifier.parameters()

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=weights)
        for p in model.parameters():
            p.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        params = model.classifier[1].parameters()

    elif model_name == "vgg16":
        model = models.vgg16(weights=weights)
        for p in model.parameters():
            p.requires_grad = False
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
        params = model.classifier[6].parameters()

    model = model.to(device)
    optimizer = optim.Adam(params, lr=LR_PHASE_1)
    return model, optimizer

# Training pipeline
kfold = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
models_to_run = ["resnet18", "densenet121", "mobilenet_v2", "vgg16"]

results_table = []

for model_name in models_to_run:
    print("\n" + "="*50)
    print("RUNNING MODEL:", model_name.upper())
    print("="*50)

    best_fold_val_acc = 0
    best_fold_val_loss = None
    best_test_acc = None
    best_test_loss = None
    best_params = None
    unique_correct = None
    unique_errors = None

    labels = train_df["label_idx"].tolist()

    for fold, (train_idx, val_idx) in enumerate(kfold.split(train_df, labels)):
        print(f"\n--- FOLD {fold+1}/{K_FOLDS} ---")

        train_subset = Subset(train_dataset, train_idx)
        val_subset = Subset(train_dataset, val_idx)

        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

        criterion = nn.CrossEntropyLoss()
        model, optimizer = get_model_and_optimizer(model_name, NUM_CLASSES_TO_USE, device)

        num_params = sum(p.numel() for p in model.parameters())

        # PHASE 1: Train head
        for epoch in range(EPOCHS_PHASE_1):
            model.train()
            for imgs, lbls in train_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
                loss.backward()
                optimizer.step()

        # PHASE 2: Fine tune
        for p in model.parameters():
            p.requires_grad = True
        optimizer = optim.Adam(model.parameters(), lr=LR_PHASE_2)

        best_this_fold_acc = 0
        best_this_fold_loss = None

        for epoch in range(EPOCHS_PHASE_2):
            model.train()
            for imgs, lbls in train_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
                loss.backward()
                optimizer.step()

            # VALIDATION
            model.eval()
            val_loss_sum = 0
            correct = 0
            total = 0

            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    out = model(imgs)
                    loss = criterion(out, lbls)
                    val_loss_sum += loss.item() * lbls.size(0)
                    _, pred = torch.max(out, 1)
                    correct += (pred == lbls).sum().item()
                    total += lbls.size(0)

            val_loss = val_loss_sum / total
            val_acc = 100 * correct / total

            if val_acc > best_this_fold_acc:
                best_this_fold_acc = val_acc
                best_this_fold_loss = val_loss

        print(f"Fold {fold+1}  Best Val Acc = {best_this_fold_acc:.2f}%")

        # CHECK IF THIS FOLD IS THE BEST
        if best_this_fold_acc > best_fold_val_acc:
            best_fold_val_acc = best_this_fold_acc
            best_fold_val_loss = best_this_fold_loss
            best_params = num_params

            # RUN TEST SET
            model.eval()
            all_preds = []
            all_labels = []
            test_loss_sum = 0
            total = 0

            with torch.no_grad():
                for imgs, lbls in test_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    out = model(imgs)
                    loss = criterion(out, lbls)
                    test_loss_sum += loss.item() * lbls.size(0)
                    _, pred = torch.max(out, 1)
                    all_preds.extend(pred.cpu().numpy())
                    all_labels.extend(lbls.cpu().numpy())
                    total += lbls.size(0)

            best_test_loss = test_loss_sum / total
            best_test_acc = 100 * sum(np.array(all_preds) == np.array(all_labels)) / total

            # UNIQUE CORRECT / ERRORS
            correct_ids = {i for i, (p, t) in enumerate(zip(all_preds, all_labels)) if p == t}
            error_ids = {i for i, (p, t) in enumerate(zip(all_preds, all_labels)) if p != t}
            unique_correct = len(correct_ids)
            unique_errors = len(error_ids)

    # saving results
    results_table.append({
        "model": model_name,
        "num_parameters": best_params,
        "val_loss": best_fold_val_loss,
        "val_accuracy": best_fold_val_acc,
        "test_loss": best_test_loss,
        "test_accuracy": best_test_acc,
        "unique_correct": unique_correct,
        "unique_errors": unique_errors
    })





**3c**

In [ ]:
final_table = pd.DataFrame(results_table)
print("FINAL MODEL RESULTS:")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
final_table

**3d**

In [ ]:
print(f"Dataset prepared: {len(df)} images of known whales.")
print(f"Number of classes used: {len(top_classes)}")

#resizing the data
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = WhaleDataset(df, TRAIN_DIR, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

# Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = models.densenet121(pretrained=True)
model = model.to(device)
model.eval()

# Extract features from the layer before the classifier
# We also add an AdaptiveAvgPool to ensure we get a fixed size vector (1024)
feature_extractor = nn.Sequential(
    model.features,
    nn.ReLU(inplace=True),
    nn.AdaptiveAvgPool2d((1, 1))
)

# Feature extraction loop
all_features = []
all_labels = []

print("Starting feature extraction (this may take ~5 mins)...")

with torch.no_grad():
    # 'tqdm' adds a progress bar so you can see it moving
    for images, labels in tqdm(dataloader, desc="Extracting"):
        images = images.to(device)

        features = feature_extractor(images)
        features = features.view(features.size(0), -1) # Flatten to (Batch, 1024)

        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())

X = np.concatenate(all_features, axis=0)
y = np.concatenate(all_labels, axis=0)

print(f"\nOriginal Feature Shape: {X.shape}")

# PCA & RANDOM FOREST (The Speed Fix)
print("Reducing dimensionality with PCA...")
# Reduce 1024 features -> 50 features to speed up Random Forest
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X)
print(f"Reduced Feature Shape: {X_pca.shape}")

print("Training Random Forest with Cross-Validation...")
# n_jobs=-1 uses all CPU cores
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Run 5-Fold CV
scores = cross_val_score(clf, X_pca, y, cv=5, scoring='accuracy')

print("\n" + "="*30)
print("RESULTS")
print("="*30)
print(f"5-Fold CV Accuracy: {scores}")
print(f"Mean Accuracy: {np.mean(scores):.4f}")

The result of this algorithm is better then our original model but it performs worse then using the transfer learning method

**3e**

In [ ]:
final_results = pd.DataFrame({
    "Experiment #": ["resnet18", "densenet121", "mobilenet_v2", "vgg16", "random forest with feature extraction"],
    "val-Loss": ["0.246215", "0.189568", "0.394751", "0.376642", "random forest doesn't use a loss function"],
    "val-Accuracy": ["95.035461%", "97.183099%", "92.957746%", "94.366197%","71.28%"],
    "Optimizer": ["Adam", "Adam", "Adam", "Adam", "random forest doesn't use an optimizer"],
    "augmentation": ["Resize to 224, 224 and normalize", "Resize to 224, 224 and normalize", "Resize to 224, 224 and normalize", "Resize to 224, 224 and normalize", "Resize to 224, 224 and normalize"]

})

final_results

#Part 4 - summary

At the beginning we made a simple CNN architecture with 5fold validation and 20 epochs this led to around 60% accuracy. We later tried to augment some of the samples in the train set using RandomRotation and ColorJitter libraries and that did not improve our model. We later implemented inference-time-augmentation which improved our model by approximately 2%, which was not a big improvement. We assume that the augmentation did not improve our model because it is too small. We then used transfer learning with 4 models resnet18, densenet121, mobilenet_v2, vgg16 this improved our model by over 30%, the accuracies were between 90% and 98% in the validation and between 83% and 92% in the test this led to the conclusion that these models that were trained with many images and with a lot of parameters can extract certain features of an image so even if we only change the last layer. The machine learning model that we did with the features extracted with densenet121 had an accuracy of 73.69% which is still better than the CNN model we did from scratch.